In [7]:

import sys
from pathlib import Path

import nibabel as nib
import numpy as np
import torch

# --------------------------------------------------
# Project paths
# --------------------------------------------------

ROOT = Path(r"D:\PS07_CT2MRI")

sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "dataset"))

from resunet import ResidualUNet
from normalise import normalise_ct


# --------------------------------------------------
# Configuration
# --------------------------------------------------

PATIENT_ID = "P2658"

INPUT_DIR = ROOT / "dataset" / "volumes" / PATIENT_ID
CHECKPOINT = ROOT / "models" / "baseline_resunet_l1.pth"

OUTPUT_DIR = ROOT / "outputs" / "reliability_test" / PATIENT_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


# --------------------------------------------------
# Load CT
# --------------------------------------------------

ct_img = nib.load(str(INPUT_DIR / "ct.nii.gz"))

ct = np.asarray(ct_img.dataobj, dtype=np.float32)
ct = normalise_ct(ct)

if ct.ndim != 3:
    raise ValueError(f"Expected a 3D CT volume, got shape {ct.shape}")

X, Y, Z = ct.shape

print("Patient:", PATIENT_ID)
print("CT shape:", ct.shape)
print("Device:", DEVICE)


# --------------------------------------------------
# Load trained model
# --------------------------------------------------

model = ResidualUNet(
    in_channels=3,
    out_channels=1,
    base_channels=16
).to(DEVICE)

checkpoint = torch.load(
    CHECKPOINT,
    map_location=DEVICE,
    weights_only=False
)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Loaded checkpoint epoch:", checkpoint["epoch"])


# --------------------------------------------------
# Four-view TTA prediction
# --------------------------------------------------

predictions = np.zeros(
    (4, X, Y, Z),
    dtype=np.float32
)

with torch.inference_mode():

    # Process EVERY slice, including boundary slices
    for z in range(Z):

        # Use nearest available slices at volume boundaries
        previous_z = max(z - 1, 0)
        next_z = min(z + 1, Z - 1)

        ct_input = np.stack(
            [
                ct[:, :, previous_z],
                ct[:, :, z],
                ct[:, :, next_z]
            ],
            axis=0
        )

        x = torch.from_numpy(
            ct_input.copy()
        ).unsqueeze(0).to(DEVICE)

        # Original prediction
        p0 = model(x)

        # Horizontal flip
        flipped_input_h = torch.flip(x, dims=[3])
        p1 = model(flipped_input_h)
        p1 = torch.flip(p1, dims=[3])

        # Vertical flip
        flipped_input_v = torch.flip(x, dims=[2])
        p2 = model(flipped_input_v)
        p2 = torch.flip(p2, dims=[2])

        # Horizontal + vertical flip
        flipped_input_both = torch.flip(x, dims=[2, 3])
        p3 = model(flipped_input_both)
        p3 = torch.flip(p3, dims=[2, 3])

        # Store predictions
        predictions[0, :, :, z] = p0[0, 0].cpu().numpy()
        predictions[1, :, :, z] = p1[0, 0].cpu().numpy()
        predictions[2, :, :, z] = p2[0, 0].cpu().numpy()
        predictions[3, :, :, z] = p3[0, 0].cpu().numpy()

        if (z + 1) % 20 == 0 or z == Z - 1:
            print(f"Processed slice {z + 1}/{Z}")


# --------------------------------------------------
# Mean prediction and uncertainty
# --------------------------------------------------

# Average of the four aligned predictions
mean_prediction = predictions.mean(axis=0)

# Standard deviation across the four predictions
uncertainty = predictions.std(
    axis=0,
    ddof=0
)

# Normalize uncertainty to [0, 1]
max_uncertainty = float(uncertainty.max())

if max_uncertainty > 0:
    uncertainty_normalized = uncertainty / max_uncertainty
else:
    uncertainty_normalized = np.zeros_like(uncertainty)

mean_prediction = mean_prediction.astype(np.float32)
uncertainty_normalized = uncertainty_normalized.astype(np.float32)


# --------------------------------------------------
# Validate outputs
# --------------------------------------------------

if not np.isfinite(mean_prediction).all():
    raise ValueError("Mean MRI contains NaN or infinite values.")

if not np.isfinite(uncertainty_normalized).all():
    raise ValueError("Uncertainty map contains NaN or infinite values.")

if mean_prediction.shape != ct.shape:
    raise ValueError("Mean MRI shape does not match CT shape.")

if uncertainty_normalized.shape != ct.shape:
    raise ValueError("Uncertainty shape does not match CT shape.")


# --------------------------------------------------
# Save NIfTI outputs
# --------------------------------------------------

output_header = ct_img.header.copy()
output_header.set_data_dtype(np.float32)

mean_img = nib.Nifti1Image(
    mean_prediction,
    ct_img.affine,
    output_header
)

uncertainty_img = nib.Nifti1Image(
    uncertainty_normalized,
    ct_img.affine,
    output_header
)

nib.save(
    mean_img,
    str(OUTPUT_DIR / "tta_mean_mri.nii.gz")
)

nib.save(
    uncertainty_img,
    str(OUTPUT_DIR / "tta_uncertainty.nii.gz")
)


# --------------------------------------------------
# Summary
# --------------------------------------------------

print("\n========== TTA RESULTS ==========")

print("Patient:", PATIENT_ID)
print("Mean MRI shape:", mean_prediction.shape)

print(
    "Mean prediction range:",
    float(mean_prediction.min()),
    float(mean_prediction.max())
)

print(
    "Raw uncertainty range:",
    float(uncertainty.min()),
    float(uncertainty.max())
)

print(
    "Normalized uncertainty range:",
    float(uncertainty_normalized.min()),
    float(uncertainty_normalized.max())
)

print(
    "First slice mean:",
    float(mean_prediction[:, :, 0].mean())
)

print(
    "Last slice mean:",
    float(mean_prediction[:, :, -1].mean())
)

print(
    "All predictions finite:",
    np.isfinite(mean_prediction).all()
)

print(
    "All uncertainty values finite:",
    np.isfinite(uncertainty_normalized).all()
)

print("Saved:", OUTPUT_DIR)
print("TTA uncertainty generation completed successfully.")


Patient: P2658
CT shape: (192, 224, 192)
Device: cuda
Loaded checkpoint epoch: 10
Processed slice 20/192
Processed slice 40/192
Processed slice 60/192
Processed slice 80/192
Processed slice 100/192
Processed slice 120/192
Processed slice 140/192
Processed slice 160/192
Processed slice 180/192
Processed slice 192/192

========== TTA RESULTS ==========
Patient: P2658
Mean MRI shape: (192, 224, 192)
Mean prediction range: 1.9050399482196077e-14 0.9203023314476013
Raw uncertainty range: 7.54075655271056e-15 0.42996302247047424
Normalized uncertainty range: 1.7538150903881507e-14 1.0
First slice mean: 0.008483635261654854
Last slice mean: 0.008483635261654854
All predictions finite: True
All uncertainty values finite: True
Saved: D:\PS07_CT2MRI\outputs\reliability_test\P2658
TTA uncertainty generation completed successfully.


In [8]:

import sys
from pathlib import Path

import nibabel as nib
import numpy as np
import torch

# --------------------------------------------------
# Project paths
# --------------------------------------------------

ROOT = Path(r"D:\PS07_CT2MRI")

sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "dataset"))

from resunet import ResidualUNet
from normalise import normalise_ct


# --------------------------------------------------
# Configuration
# --------------------------------------------------

PATIENT_ID = "P6123"

INPUT_DIR = ROOT / "dataset" / "volumes" / PATIENT_ID
CHECKPOINT = ROOT / "models" / "baseline_resunet_l1.pth"

OUTPUT_DIR = ROOT / "outputs" / "reliability_test" / PATIENT_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


# --------------------------------------------------
# Load CT
# --------------------------------------------------

ct_img = nib.load(str(INPUT_DIR / "ct.nii.gz"))

ct = np.asarray(ct_img.dataobj, dtype=np.float32)
ct = normalise_ct(ct)

if ct.ndim != 3:
    raise ValueError(f"Expected a 3D CT volume, got shape {ct.shape}")

X, Y, Z = ct.shape

print("Patient:", PATIENT_ID)
print("CT shape:", ct.shape)
print("Device:", DEVICE)


# --------------------------------------------------
# Load trained model
# --------------------------------------------------

model = ResidualUNet(
    in_channels=3,
    out_channels=1,
    base_channels=16
).to(DEVICE)

checkpoint = torch.load(
    CHECKPOINT,
    map_location=DEVICE,
    weights_only=False
)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Loaded checkpoint epoch:", checkpoint["epoch"])


# --------------------------------------------------
# Four-view TTA prediction
# --------------------------------------------------

predictions = np.zeros(
    (4, X, Y, Z),
    dtype=np.float32
)

with torch.inference_mode():

    # Process EVERY slice, including boundary slices
    for z in range(Z):

        # Use nearest available slices at volume boundaries
        previous_z = max(z - 1, 0)
        next_z = min(z + 1, Z - 1)

        ct_input = np.stack(
            [
                ct[:, :, previous_z],
                ct[:, :, z],
                ct[:, :, next_z]
            ],
            axis=0
        )

        x = torch.from_numpy(
            ct_input.copy()
        ).unsqueeze(0).to(DEVICE)

        # Original prediction
        p0 = model(x)

        # Horizontal flip
        flipped_input_h = torch.flip(x, dims=[3])
        p1 = model(flipped_input_h)
        p1 = torch.flip(p1, dims=[3])

        # Vertical flip
        flipped_input_v = torch.flip(x, dims=[2])
        p2 = model(flipped_input_v)
        p2 = torch.flip(p2, dims=[2])

        # Horizontal + vertical flip
        flipped_input_both = torch.flip(x, dims=[2, 3])
        p3 = model(flipped_input_both)
        p3 = torch.flip(p3, dims=[2, 3])

        # Store predictions
        predictions[0, :, :, z] = p0[0, 0].cpu().numpy()
        predictions[1, :, :, z] = p1[0, 0].cpu().numpy()
        predictions[2, :, :, z] = p2[0, 0].cpu().numpy()
        predictions[3, :, :, z] = p3[0, 0].cpu().numpy()

        if (z + 1) % 20 == 0 or z == Z - 1:
            print(f"Processed slice {z + 1}/{Z}")


# --------------------------------------------------
# Mean prediction and uncertainty
# --------------------------------------------------

# Average of the four aligned predictions
mean_prediction = predictions.mean(axis=0)

# Standard deviation across the four predictions
uncertainty = predictions.std(
    axis=0,
    ddof=0
)

# Normalize uncertainty to [0, 1]
max_uncertainty = float(uncertainty.max())

if max_uncertainty > 0:
    uncertainty_normalized = uncertainty / max_uncertainty
else:
    uncertainty_normalized = np.zeros_like(uncertainty)

mean_prediction = mean_prediction.astype(np.float32)
uncertainty_normalized = uncertainty_normalized.astype(np.float32)


# --------------------------------------------------
# Validate outputs
# --------------------------------------------------

if not np.isfinite(mean_prediction).all():
    raise ValueError("Mean MRI contains NaN or infinite values.")

if not np.isfinite(uncertainty_normalized).all():
    raise ValueError("Uncertainty map contains NaN or infinite values.")

if mean_prediction.shape != ct.shape:
    raise ValueError("Mean MRI shape does not match CT shape.")

if uncertainty_normalized.shape != ct.shape:
    raise ValueError("Uncertainty shape does not match CT shape.")


# --------------------------------------------------
# Save NIfTI outputs
# --------------------------------------------------

output_header = ct_img.header.copy()
output_header.set_data_dtype(np.float32)

mean_img = nib.Nifti1Image(
    mean_prediction,
    ct_img.affine,
    output_header
)

uncertainty_img = nib.Nifti1Image(
    uncertainty_normalized,
    ct_img.affine,
    output_header
)

nib.save(
    mean_img,
    str(OUTPUT_DIR / "tta_mean_mri.nii.gz")
)

nib.save(
    uncertainty_img,
    str(OUTPUT_DIR / "tta_uncertainty.nii.gz")
)


# --------------------------------------------------
# Summary
# --------------------------------------------------

print("\n========== TTA RESULTS ==========")

print("Patient:", PATIENT_ID)
print("Mean MRI shape:", mean_prediction.shape)

print(
    "Mean prediction range:",
    float(mean_prediction.min()),
    float(mean_prediction.max())
)

print(
    "Raw uncertainty range:",
    float(uncertainty.min()),
    float(uncertainty.max())
)

print(
    "Normalized uncertainty range:",
    float(uncertainty_normalized.min()),
    float(uncertainty_normalized.max())
)

print(
    "First slice mean:",
    float(mean_prediction[:, :, 0].mean())
)

print(
    "Last slice mean:",
    float(mean_prediction[:, :, -1].mean())
)

print(
    "All predictions finite:",
    np.isfinite(mean_prediction).all()
)

print(
    "All uncertainty values finite:",
    np.isfinite(uncertainty_normalized).all()
)

print("Saved:", OUTPUT_DIR)
print("TTA uncertainty generation completed successfully.")


Patient: P6123
CT shape: (192, 224, 192)
Device: cuda
Loaded checkpoint epoch: 10
Processed slice 20/192
Processed slice 40/192
Processed slice 60/192
Processed slice 80/192
Processed slice 100/192
Processed slice 120/192
Processed slice 140/192
Processed slice 160/192
Processed slice 180/192
Processed slice 192/192

========== TTA RESULTS ==========
Patient: P6123
Mean MRI shape: (192, 224, 192)
Mean prediction range: 6.84637025891166e-13 0.9153159856796265
Raw uncertainty range: 6.921921260998054e-13 0.43456053733825684
Normalized uncertainty range: 1.5928555132868616e-12 1.0
First slice mean: 0.23041768372058868
Last slice mean: 0.008483635261654854
All predictions finite: True
All uncertainty values finite: True
Saved: D:\PS07_CT2MRI\outputs\reliability_test\P6123
TTA uncertainty generation completed successfully.


In [9]:

import sys
from pathlib import Path

import nibabel as nib
import numpy as np
import torch

# --------------------------------------------------
# Project paths
# --------------------------------------------------

ROOT = Path(r"D:\PS07_CT2MRI")

sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "dataset"))

from resunet import ResidualUNet
from normalise import normalise_ct


# --------------------------------------------------
# Configuration
# --------------------------------------------------

PATIENT_ID = "P7363"

INPUT_DIR = ROOT / "dataset" / "volumes" / PATIENT_ID
CHECKPOINT = ROOT / "models" / "baseline_resunet_l1.pth"

OUTPUT_DIR = ROOT / "outputs" / "reliability_test" / PATIENT_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


# --------------------------------------------------
# Load CT
# --------------------------------------------------

ct_img = nib.load(str(INPUT_DIR / "ct.nii.gz"))

ct = np.asarray(ct_img.dataobj, dtype=np.float32)
ct = normalise_ct(ct)

if ct.ndim != 3:
    raise ValueError(f"Expected a 3D CT volume, got shape {ct.shape}")

X, Y, Z = ct.shape

print("Patient:", PATIENT_ID)
print("CT shape:", ct.shape)
print("Device:", DEVICE)


# --------------------------------------------------
# Load trained model
# --------------------------------------------------

model = ResidualUNet(
    in_channels=3,
    out_channels=1,
    base_channels=16
).to(DEVICE)

checkpoint = torch.load(
    CHECKPOINT,
    map_location=DEVICE,
    weights_only=False
)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Loaded checkpoint epoch:", checkpoint["epoch"])


# --------------------------------------------------
# Four-view TTA prediction
# --------------------------------------------------

predictions = np.zeros(
    (4, X, Y, Z),
    dtype=np.float32
)

with torch.inference_mode():

    # Process EVERY slice, including boundary slices
    for z in range(Z):

        # Use nearest available slices at volume boundaries
        previous_z = max(z - 1, 0)
        next_z = min(z + 1, Z - 1)

        ct_input = np.stack(
            [
                ct[:, :, previous_z],
                ct[:, :, z],
                ct[:, :, next_z]
            ],
            axis=0
        )

        x = torch.from_numpy(
            ct_input.copy()
        ).unsqueeze(0).to(DEVICE)

        # Original prediction
        p0 = model(x)

        # Horizontal flip
        flipped_input_h = torch.flip(x, dims=[3])
        p1 = model(flipped_input_h)
        p1 = torch.flip(p1, dims=[3])

        # Vertical flip
        flipped_input_v = torch.flip(x, dims=[2])
        p2 = model(flipped_input_v)
        p2 = torch.flip(p2, dims=[2])

        # Horizontal + vertical flip
        flipped_input_both = torch.flip(x, dims=[2, 3])
        p3 = model(flipped_input_both)
        p3 = torch.flip(p3, dims=[2, 3])

        # Store predictions
        predictions[0, :, :, z] = p0[0, 0].cpu().numpy()
        predictions[1, :, :, z] = p1[0, 0].cpu().numpy()
        predictions[2, :, :, z] = p2[0, 0].cpu().numpy()
        predictions[3, :, :, z] = p3[0, 0].cpu().numpy()

        if (z + 1) % 20 == 0 or z == Z - 1:
            print(f"Processed slice {z + 1}/{Z}")


# --------------------------------------------------
# Mean prediction and uncertainty
# --------------------------------------------------

# Average of the four aligned predictions
mean_prediction = predictions.mean(axis=0)

# Standard deviation across the four predictions
uncertainty = predictions.std(
    axis=0,
    ddof=0
)

# Normalize uncertainty to [0, 1]
max_uncertainty = float(uncertainty.max())

if max_uncertainty > 0:
    uncertainty_normalized = uncertainty / max_uncertainty
else:
    uncertainty_normalized = np.zeros_like(uncertainty)

mean_prediction = mean_prediction.astype(np.float32)
uncertainty_normalized = uncertainty_normalized.astype(np.float32)


# --------------------------------------------------
# Validate outputs
# --------------------------------------------------

if not np.isfinite(mean_prediction).all():
    raise ValueError("Mean MRI contains NaN or infinite values.")

if not np.isfinite(uncertainty_normalized).all():
    raise ValueError("Uncertainty map contains NaN or infinite values.")

if mean_prediction.shape != ct.shape:
    raise ValueError("Mean MRI shape does not match CT shape.")

if uncertainty_normalized.shape != ct.shape:
    raise ValueError("Uncertainty shape does not match CT shape.")


# --------------------------------------------------
# Save NIfTI outputs
# --------------------------------------------------

output_header = ct_img.header.copy()
output_header.set_data_dtype(np.float32)

mean_img = nib.Nifti1Image(
    mean_prediction,
    ct_img.affine,
    output_header
)

uncertainty_img = nib.Nifti1Image(
    uncertainty_normalized,
    ct_img.affine,
    output_header
)

nib.save(
    mean_img,
    str(OUTPUT_DIR / "tta_mean_mri.nii.gz")
)

nib.save(
    uncertainty_img,
    str(OUTPUT_DIR / "tta_uncertainty.nii.gz")
)


# --------------------------------------------------
# Summary
# --------------------------------------------------

print("\n========== TTA RESULTS ==========")

print("Patient:", PATIENT_ID)
print("Mean MRI shape:", mean_prediction.shape)

print(
    "Mean prediction range:",
    float(mean_prediction.min()),
    float(mean_prediction.max())
)

print(
    "Raw uncertainty range:",
    float(uncertainty.min()),
    float(uncertainty.max())
)

print(
    "Normalized uncertainty range:",
    float(uncertainty_normalized.min()),
    float(uncertainty_normalized.max())
)

print(
    "First slice mean:",
    float(mean_prediction[:, :, 0].mean())
)

print(
    "Last slice mean:",
    float(mean_prediction[:, :, -1].mean())
)

print(
    "All predictions finite:",
    np.isfinite(mean_prediction).all()
)

print(
    "All uncertainty values finite:",
    np.isfinite(uncertainty_normalized).all()
)

print("Saved:", OUTPUT_DIR)
print("TTA uncertainty generation completed successfully.")


Patient: P7363
CT shape: (192, 224, 192)
Device: cuda
Loaded checkpoint epoch: 10
Processed slice 20/192
Processed slice 40/192
Processed slice 60/192
Processed slice 80/192
Processed slice 100/192
Processed slice 120/192
Processed slice 140/192
Processed slice 160/192
Processed slice 180/192
Processed slice 192/192

========== TTA RESULTS ==========
Patient: P7363
Mean MRI shape: (192, 224, 192)
Mean prediction range: 3.834249584500071e-11 0.8990160226821899
Raw uncertainty range: 3.8345979169740474e-11 0.3997287452220917
Normalized uncertainty range: 9.593000005470032e-11 1.0
First slice mean: 0.1516287624835968
Last slice mean: 0.008483635261654854
All predictions finite: True
All uncertainty values finite: True
Saved: D:\PS07_CT2MRI\outputs\reliability_test\P7363
TTA uncertainty generation completed successfully.


In [10]:

import sys
from pathlib import Path

import nibabel as nib
import numpy as np
import torch

# --------------------------------------------------
# Project paths
# --------------------------------------------------

ROOT = Path(r"D:\PS07_CT2MRI")

sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "dataset"))

from resunet import ResidualUNet
from normalise import normalise_ct


# --------------------------------------------------
# Configuration
# --------------------------------------------------

PATIENT_ID = "P7480"

INPUT_DIR = ROOT / "dataset" / "volumes" / PATIENT_ID
CHECKPOINT = ROOT / "models" / "baseline_resunet_l1.pth"

OUTPUT_DIR = ROOT / "outputs" / "reliability_test" / PATIENT_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


# --------------------------------------------------
# Load CT
# --------------------------------------------------

ct_img = nib.load(str(INPUT_DIR / "ct.nii.gz"))

ct = np.asarray(ct_img.dataobj, dtype=np.float32)
ct = normalise_ct(ct)

if ct.ndim != 3:
    raise ValueError(f"Expected a 3D CT volume, got shape {ct.shape}")

X, Y, Z = ct.shape

print("Patient:", PATIENT_ID)
print("CT shape:", ct.shape)
print("Device:", DEVICE)


# --------------------------------------------------
# Load trained model
# --------------------------------------------------

model = ResidualUNet(
    in_channels=3,
    out_channels=1,
    base_channels=16
).to(DEVICE)

checkpoint = torch.load(
    CHECKPOINT,
    map_location=DEVICE,
    weights_only=False
)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Loaded checkpoint epoch:", checkpoint["epoch"])


# --------------------------------------------------
# Four-view TTA prediction
# --------------------------------------------------

predictions = np.zeros(
    (4, X, Y, Z),
    dtype=np.float32
)

with torch.inference_mode():

    # Process EVERY slice, including boundary slices
    for z in range(Z):

        # Use nearest available slices at volume boundaries
        previous_z = max(z - 1, 0)
        next_z = min(z + 1, Z - 1)

        ct_input = np.stack(
            [
                ct[:, :, previous_z],
                ct[:, :, z],
                ct[:, :, next_z]
            ],
            axis=0
        )

        x = torch.from_numpy(
            ct_input.copy()
        ).unsqueeze(0).to(DEVICE)

        # Original prediction
        p0 = model(x)

        # Horizontal flip
        flipped_input_h = torch.flip(x, dims=[3])
        p1 = model(flipped_input_h)
        p1 = torch.flip(p1, dims=[3])

        # Vertical flip
        flipped_input_v = torch.flip(x, dims=[2])
        p2 = model(flipped_input_v)
        p2 = torch.flip(p2, dims=[2])

        # Horizontal + vertical flip
        flipped_input_both = torch.flip(x, dims=[2, 3])
        p3 = model(flipped_input_both)
        p3 = torch.flip(p3, dims=[2, 3])

        # Store predictions
        predictions[0, :, :, z] = p0[0, 0].cpu().numpy()
        predictions[1, :, :, z] = p1[0, 0].cpu().numpy()
        predictions[2, :, :, z] = p2[0, 0].cpu().numpy()
        predictions[3, :, :, z] = p3[0, 0].cpu().numpy()

        if (z + 1) % 20 == 0 or z == Z - 1:
            print(f"Processed slice {z + 1}/{Z}")


# --------------------------------------------------
# Mean prediction and uncertainty
# --------------------------------------------------

# Average of the four aligned predictions
mean_prediction = predictions.mean(axis=0)

# Standard deviation across the four predictions
uncertainty = predictions.std(
    axis=0,
    ddof=0
)

# Normalize uncertainty to [0, 1]
max_uncertainty = float(uncertainty.max())

if max_uncertainty > 0:
    uncertainty_normalized = uncertainty / max_uncertainty
else:
    uncertainty_normalized = np.zeros_like(uncertainty)

mean_prediction = mean_prediction.astype(np.float32)
uncertainty_normalized = uncertainty_normalized.astype(np.float32)


# --------------------------------------------------
# Validate outputs
# --------------------------------------------------

if not np.isfinite(mean_prediction).all():
    raise ValueError("Mean MRI contains NaN or infinite values.")

if not np.isfinite(uncertainty_normalized).all():
    raise ValueError("Uncertainty map contains NaN or infinite values.")

if mean_prediction.shape != ct.shape:
    raise ValueError("Mean MRI shape does not match CT shape.")

if uncertainty_normalized.shape != ct.shape:
    raise ValueError("Uncertainty shape does not match CT shape.")


# --------------------------------------------------
# Save NIfTI outputs
# --------------------------------------------------

output_header = ct_img.header.copy()
output_header.set_data_dtype(np.float32)

mean_img = nib.Nifti1Image(
    mean_prediction,
    ct_img.affine,
    output_header
)

uncertainty_img = nib.Nifti1Image(
    uncertainty_normalized,
    ct_img.affine,
    output_header
)

nib.save(
    mean_img,
    str(OUTPUT_DIR / "tta_mean_mri.nii.gz")
)

nib.save(
    uncertainty_img,
    str(OUTPUT_DIR / "tta_uncertainty.nii.gz")
)


# --------------------------------------------------
# Summary
# --------------------------------------------------

print("\n========== TTA RESULTS ==========")

print("Patient:", PATIENT_ID)
print("Mean MRI shape:", mean_prediction.shape)

print(
    "Mean prediction range:",
    float(mean_prediction.min()),
    float(mean_prediction.max())
)

print(
    "Raw uncertainty range:",
    float(uncertainty.min()),
    float(uncertainty.max())
)

print(
    "Normalized uncertainty range:",
    float(uncertainty_normalized.min()),
    float(uncertainty_normalized.max())
)

print(
    "First slice mean:",
    float(mean_prediction[:, :, 0].mean())
)

print(
    "Last slice mean:",
    float(mean_prediction[:, :, -1].mean())
)

print(
    "All predictions finite:",
    np.isfinite(mean_prediction).all()
)

print(
    "All uncertainty values finite:",
    np.isfinite(uncertainty_normalized).all()
)

print("Saved:", OUTPUT_DIR)
print("TTA uncertainty generation completed successfully.")


Patient: P7480
CT shape: (192, 224, 192)
Device: cuda
Loaded checkpoint epoch: 10
Processed slice 20/192
Processed slice 40/192
Processed slice 60/192
Processed slice 80/192
Processed slice 100/192
Processed slice 120/192
Processed slice 140/192
Processed slice 160/192
Processed slice 180/192
Processed slice 192/192

========== TTA RESULTS ==========
Patient: P7480
Mean MRI shape: (192, 224, 192)
Mean prediction range: 2.221669723190267e-11 0.9205033779144287
Raw uncertainty range: 2.2571961660888817e-11 0.4382026791572571
Normalized uncertainty range: 5.1510323784143e-11 1.0
First slice mean: 0.008483635261654854
Last slice mean: 0.008483635261654854
All predictions finite: True
All uncertainty values finite: True
Saved: D:\PS07_CT2MRI\outputs\reliability_test\P7480
TTA uncertainty generation completed successfully.


In [11]:

import sys
from pathlib import Path

import nibabel as nib
import numpy as np
import torch

# --------------------------------------------------
# Project paths
# --------------------------------------------------

ROOT = Path(r"D:\PS07_CT2MRI")

sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "dataset"))

from resunet import ResidualUNet
from normalise import normalise_ct


# --------------------------------------------------
# Configuration
# --------------------------------------------------

PATIENT_ID = "P8947"

INPUT_DIR = ROOT / "dataset" / "volumes" / PATIENT_ID
CHECKPOINT = ROOT / "models" / "baseline_resunet_l1.pth"

OUTPUT_DIR = ROOT / "outputs" / "reliability_test" / PATIENT_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


# --------------------------------------------------
# Load CT
# --------------------------------------------------

ct_img = nib.load(str(INPUT_DIR / "ct.nii.gz"))

ct = np.asarray(ct_img.dataobj, dtype=np.float32)
ct = normalise_ct(ct)

if ct.ndim != 3:
    raise ValueError(f"Expected a 3D CT volume, got shape {ct.shape}")

X, Y, Z = ct.shape

print("Patient:", PATIENT_ID)
print("CT shape:", ct.shape)
print("Device:", DEVICE)


# --------------------------------------------------
# Load trained model
# --------------------------------------------------

model = ResidualUNet(
    in_channels=3,
    out_channels=1,
    base_channels=16
).to(DEVICE)

checkpoint = torch.load(
    CHECKPOINT,
    map_location=DEVICE,
    weights_only=False
)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Loaded checkpoint epoch:", checkpoint["epoch"])


# --------------------------------------------------
# Four-view TTA prediction
# --------------------------------------------------

predictions = np.zeros(
    (4, X, Y, Z),
    dtype=np.float32
)

with torch.inference_mode():

    # Process EVERY slice, including boundary slices
    for z in range(Z):

        # Use nearest available slices at volume boundaries
        previous_z = max(z - 1, 0)
        next_z = min(z + 1, Z - 1)

        ct_input = np.stack(
            [
                ct[:, :, previous_z],
                ct[:, :, z],
                ct[:, :, next_z]
            ],
            axis=0
        )

        x = torch.from_numpy(
            ct_input.copy()
        ).unsqueeze(0).to(DEVICE)

        # Original prediction
        p0 = model(x)

        # Horizontal flip
        flipped_input_h = torch.flip(x, dims=[3])
        p1 = model(flipped_input_h)
        p1 = torch.flip(p1, dims=[3])

        # Vertical flip
        flipped_input_v = torch.flip(x, dims=[2])
        p2 = model(flipped_input_v)
        p2 = torch.flip(p2, dims=[2])

        # Horizontal + vertical flip
        flipped_input_both = torch.flip(x, dims=[2, 3])
        p3 = model(flipped_input_both)
        p3 = torch.flip(p3, dims=[2, 3])

        # Store predictions
        predictions[0, :, :, z] = p0[0, 0].cpu().numpy()
        predictions[1, :, :, z] = p1[0, 0].cpu().numpy()
        predictions[2, :, :, z] = p2[0, 0].cpu().numpy()
        predictions[3, :, :, z] = p3[0, 0].cpu().numpy()

        if (z + 1) % 20 == 0 or z == Z - 1:
            print(f"Processed slice {z + 1}/{Z}")


# --------------------------------------------------
# Mean prediction and uncertainty
# --------------------------------------------------

# Average of the four aligned predictions
mean_prediction = predictions.mean(axis=0)

# Standard deviation across the four predictions
uncertainty = predictions.std(
    axis=0,
    ddof=0
)

# Normalize uncertainty to [0, 1]
max_uncertainty = float(uncertainty.max())

if max_uncertainty > 0:
    uncertainty_normalized = uncertainty / max_uncertainty
else:
    uncertainty_normalized = np.zeros_like(uncertainty)

mean_prediction = mean_prediction.astype(np.float32)
uncertainty_normalized = uncertainty_normalized.astype(np.float32)


# --------------------------------------------------
# Validate outputs
# --------------------------------------------------

if not np.isfinite(mean_prediction).all():
    raise ValueError("Mean MRI contains NaN or infinite values.")

if not np.isfinite(uncertainty_normalized).all():
    raise ValueError("Uncertainty map contains NaN or infinite values.")

if mean_prediction.shape != ct.shape:
    raise ValueError("Mean MRI shape does not match CT shape.")

if uncertainty_normalized.shape != ct.shape:
    raise ValueError("Uncertainty shape does not match CT shape.")


# --------------------------------------------------
# Save NIfTI outputs
# --------------------------------------------------

output_header = ct_img.header.copy()
output_header.set_data_dtype(np.float32)

mean_img = nib.Nifti1Image(
    mean_prediction,
    ct_img.affine,
    output_header
)

uncertainty_img = nib.Nifti1Image(
    uncertainty_normalized,
    ct_img.affine,
    output_header
)

nib.save(
    mean_img,
    str(OUTPUT_DIR / "tta_mean_mri.nii.gz")
)

nib.save(
    uncertainty_img,
    str(OUTPUT_DIR / "tta_uncertainty.nii.gz")
)


# --------------------------------------------------
# Summary
# --------------------------------------------------

print("\n========== TTA RESULTS ==========")

print("Patient:", PATIENT_ID)
print("Mean MRI shape:", mean_prediction.shape)

print(
    "Mean prediction range:",
    float(mean_prediction.min()),
    float(mean_prediction.max())
)

print(
    "Raw uncertainty range:",
    float(uncertainty.min()),
    float(uncertainty.max())
)

print(
    "Normalized uncertainty range:",
    float(uncertainty_normalized.min()),
    float(uncertainty_normalized.max())
)

print(
    "First slice mean:",
    float(mean_prediction[:, :, 0].mean())
)

print(
    "Last slice mean:",
    float(mean_prediction[:, :, -1].mean())
)

print(
    "All predictions finite:",
    np.isfinite(mean_prediction).all()
)

print(
    "All uncertainty values finite:",
    np.isfinite(uncertainty_normalized).all()
)

print("Saved:", OUTPUT_DIR)
print("TTA uncertainty generation completed successfully.")


Patient: P8947
CT shape: (192, 224, 192)
Device: cuda
Loaded checkpoint epoch: 10
Processed slice 20/192
Processed slice 40/192
Processed slice 60/192
Processed slice 80/192
Processed slice 100/192
Processed slice 120/192
Processed slice 140/192
Processed slice 160/192
Processed slice 180/192
Processed slice 192/192

========== TTA RESULTS ==========
Patient: P8947
Mean MRI shape: (192, 224, 192)
Mean prediction range: 1.285141968971476e-13 0.9272228479385376
Raw uncertainty range: 1.3816752646514385e-13 0.4380091428756714
Normalized uncertainty range: 3.154443936961848e-13 1.0
First slice mean: 0.15918783843517303
Last slice mean: 0.008483635261654854
All predictions finite: True
All uncertainty values finite: True
Saved: D:\PS07_CT2MRI\outputs\reliability_test\P8947
TTA uncertainty generation completed successfully.
